# Protocolos de Distribución Cuántica de Claves

Librerías a utilizar:

In [1]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

## BB84

Se define la funcion para el protocolo

In [18]:
def bb84(longitud_clave=1000, con_eve=False):
    # Inicialización del simulador cuántico local
    backend = AerSimulator()
    
    # Generación de flujos de bits y bases mediante distribuciones uniformes
    #np.random.randint(low = 0, high = n-1, size = longitud_clave)
    alice_bits = np.random.randint(2, size=longitud_clave)
    alice_bases = np.random.randint(2, size=longitud_clave) # 0: Base Z, 1: Base X
    bob_bases = np.random.randint(2, size=longitud_clave)
    eve_bases = np.random.randint(2, size=longitud_clave)
    
    #Inicializacion de los resultados de Bob y circuitos
    bob_resultados = []
    circuitos = []
    
    for i in range(longitud_clave):
        # Asignación de un circuito individual de un qubit y un bit clásico para guardar la medicion
        qc = QuantumCircuit(1, 1)
        
        # Etapa de Codificación (Alice)
        #Siempre partimos de |0>, pero si Alice quiere un 1, entonces codificamos
        if alice_bits[i] == 1: 
            qc.x(0) # Inversión para bit 1
        elif alice_bits[i] == 0:
            #X|0> =|0>
            pass
        #Similarmente, si Alice quiere enviar el bit en base diagonal, codificamos
        if alice_bases[i] == 1: 
            qc.h(0) # Transición a la base diagonal (Hadamard)
            
        # Etapa de Invasión (Eve: Ataque de Interceptar y Reenviar)
        if con_eve:
            # Eve alinea su sistema de medición según el caso
            if eve_bases[i] == 1:
                qc.h(0)
            #Recordar qc.measure(indice qubit a medir, indice qubit clasico en cual escribir). Solo hay
            #Uno clasico y cuantico, entonces siempre será 0
            qc.measure(0, 0) #La medida colapsa el estado
            
            # Reenvío activo del estado colapsado
            if eve_bases[i] == 1:
                qc.h(0) # Reversión para la inyección en el canal en caso que ella usara base Hadamard
                #Esto es solo con tal de reutilizar el qubit, pero podría perfectamente enviar otra cosa
                
        # Etapa de Detección (Bob)
        if bob_bases[i] == 1:
            qc.h(0)
            
        qc.measure(0, 0)
        
        # Añadimos el circuito a la lista para procesarlo en lote
        circuitos.append(qc)
        
    # Transpilamos y ejecutamos todos los circuitos a la vez
    qc_transpiled = transpile(circuitos, backend)
    job = backend.run(qc_transpiled, shots=1, memory=True) #transpilamos, corremos una vez, y guardamos
    resultados_job = job.result() #Obtiene resultados de medidas finales
    
    # Extraemos los resultados
    for i in range(longitud_clave):
        resultado = int(resultados_job.get_memory(i)[0])
        bob_resultados.append(resultado)
        
    # Fase Clásica: Cribado (Sifting)
    bob_resultados = np.array(bob_resultados) #Forma un array con los resultados de Bob
    coincidencias = alice_bases == bob_bases #Eficiente, compara cada elemento del array
    
    # Extracción de la clave depurada
    clave_alice = alice_bits[coincidencias]
    clave_bob = bob_resultados[coincidencias]
    
    # Computación de Quantum Bit Error Rate (QBER)
    errores = np.sum(clave_alice != clave_bob)
    total_bits_utiles = len(clave_alice)
    qber = (errores / total_bits_utiles) * 100 if total_bits_utiles > 0 else 0.0
    
    print("="*60)
    print(f" SIMULACIÓN BB84 | Presencia de Eve: {con_eve}")
    print("="*60)
    print(f"Longitud de transmisión inicial: {longitud_clave} fotones")
    print(f"Longitud de clave cribada: {total_bits_utiles} bits (~50% esperado)")
    print(f"Discrepancias detectadas: {errores} bits")
    print(f"Tasa de Error Cuántico (QBER): {qber:.2f}%")
    
    return qber

Finalmente, se corre el programa, evaluando el caso con Eve y sin Eve

In [19]:
print("Protocolo BB84...")
bb84_limpio = bb84(longitud_clave=1000, con_eve=False)
print("\n")
bb84_interceptado = bb84(longitud_clave=1000, con_eve=True)

Protocolo BB84...
 SIMULACIÓN BB84 | Presencia de Eve: False
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 500 bits (~50% esperado)
Discrepancias detectadas: 0 bits
Tasa de Error Cuántico (QBER): 0.00%


 SIMULACIÓN BB84 | Presencia de Eve: True
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 516 bits (~50% esperado)
Discrepancias detectadas: 132 bits
Tasa de Error Cuántico (QBER): 25.58%


## Protocolo Seis Estados

Va a ser en gran parte similar, pero con una base extra. En tal caso, es relevante introducir una nueva compuerta lógica cuántica: Compuerta S, la cual es unitaria (como siempre), pero no hermítica, por lo que $SS^\dagger=\mathbb{I}$ pero $S\neq S^\dagger$.
$$
S = \begin{bmatrix}
1 & 0 \\
0 & i
\end{bmatrix}
\hspace{0.5 cm}
S^\dagger = \begin{bmatrix}
1 & 0 \\
0 & -i
\end{bmatrix}
$$
La mayor parte del codigo es igual:

In [11]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def six_state(longitud_clave=1000, con_eve=False):
    """
   Ahora se tiene la base estandar Z, la base de Hadamard X y la base imaginaria Y,
   Completando las dimensiones de la esfera de Bloch
    """
    backend = AerSimulator()
    
    #Bases Z=0, X=1, Y=2
    alice_bits = np.random.randint(2, size=longitud_clave)
    alice_bases = np.random.randint(3, size=longitud_clave) #Como hay una base extra, se modifica el limite superior
    bob_bases = np.random.randint(3, size=longitud_clave)
    eve_bases = np.random.randint(3, size=longitud_clave)
    
    bob_resultados = []
    circuitos = []
    
    for i in range(longitud_clave):
        qc = QuantumCircuit(1, 1)
        
        #Alice
        if alice_bits[i] == 1:
            qc.x(0)
            
        if alice_bases[i] == 1:     # Proyección hacia Base X
            qc.h(0)
        elif alice_bases[i] == 2:   # Proyección hacia Base Y
            qc.h(0)
            qc.s(0)
            
        #Eve
        if con_eve:
            if eve_bases[i] == 1:
                qc.h(0)
            #Nuevo condicional, el resto se mantiene
            elif eve_bases[i] == 2:
                qc.sdg(0) # Operación adjunta (S-dagger) para rotación inversa
                qc.h(0)
                
            qc.measure(0, 0)
            
            if eve_bases[i] == 1:
                qc.h(0)
            elif eve_bases[i] == 2:
                #Similarmente, se reversa la transformacion para enviar el qubit
                qc.h(0)
                qc.s(0)
                
        #Bob
        if bob_bases[i] == 1:
            qc.h(0)
        elif bob_bases[i] == 2:
            qc.sdg(0)
            qc.h(0)
            
        qc.measure(0, 0)
        
        circuitos.append(qc)

    qc_transpiled_list = transpile(circuitos, backend)
    job = backend.run(qc_transpiled_list, shots=1, memory=True)
    resultados_simulacion = job.result()
    
    for i in range(longitud_clave):
        bit_medido = int(resultados_simulacion.get_memory(i)[0])
        bob_resultados.append(bit_medido)
        
    # Reconciliación Rigurosa y Evaluación de Seguridad
    bob_resultados = np.array(bob_resultados)
    coincidencias = (alice_bases == bob_bases)
    
    clave_alice = alice_bits[coincidencias]
    clave_bob = bob_resultados[coincidencias]
    
    errores = np.sum(clave_alice != clave_bob)
    total_bits_utiles = len(clave_alice)
    qber = (errores / total_bits_utiles) * 100 if total_bits_utiles > 0 else 0.0
    
    print("="*60)
    print(f" SIMULACIÓN SIX-STATE | Presencia de Eve: {con_eve}")
    print("="*60)
    print(f"Longitud de transmisión inicial: {longitud_clave} fotones")
    print(f"Longitud de clave cribada: {total_bits_utiles} bits (~33.3% esperado)")
    print(f"Discrepancias detectadas: {errores} bits")
    print(f"Tasa de Error Cuántico (QBER): {qber:.2f}%")
    
    return qber

#=============== Main ===================
print("Protocolo Six-State...")
six_state_limpio = six_state(longitud_clave=1000, con_eve=False)
print("\n")
six_state_interceptado = six_state(longitud_clave=1000, con_eve=True)

Protocolo Six-State...
 SIMULACIÓN SIX-STATE | Presencia de Eve: False
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 313 bits (~33.3% esperado)
Discrepancias detectadas: 0 bits
Tasa de Error Cuántico (QBER): 0.00%


 SIMULACIÓN SIX-STATE | Presencia de Eve: True
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 339 bits (~33.3% esperado)
Discrepancias detectadas: 111 bits
Tasa de Error Cuántico (QBER): 32.74%


In [23]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def b92(longitud_clave=1000, con_eve=False):
    """
    Estados enviados por Alice: bit 0 -> |0>, bit 1 -> |+>.
    Bob mide en base Z (0) o X (1). Solo conserva medidas en |1> de cada respectiva base (es decir, |1> y |->).
    """
    backend = AerSimulator()

    # Alice solo genera bits (no elige base)
    alice_bits = np.random.randint(2, size=longitud_clave)

    # Bases de Bob y Eve (0: Z, 1: X)
    bob_bases = np.random.randint(2, size=longitud_clave)
    eve_bases = np.random.randint(2, size=longitud_clave)

    # Lista para almacenar los circuitos
    circuitos = []

    #ES un ciclo similar, pero sin los condicionales de las bases de Alice y cambiando la transformacion al bit 1 de X a H
    for i in range(longitud_clave):
        qc = QuantumCircuit(1, 1)

        # ---- Alice: codificación ----
        # |0> por defecto (bit 0). Para bit 1 aplicamos H -> |+>
        if alice_bits[i] == 1:
            qc.h(0) #En bb84 se usaba la compuerta X, ahora se usa la de Hadamard

        # ---- Eve: interceptar y reenviar (si con_eve) ----
        if con_eve:
            if eve_bases[i] == 1:      # Base X
                qc.h(0)
            qc.measure(0, 0)           # Medición colapsa el estado
            # Reconstrucción del estado medido para reenviar
            if eve_bases[i] == 1:
                qc.h(0)                # Vuelve a la base X (|+> o |->)

        # ---- Bob: medición ----
        if bob_bases[i] == 1:          # Base X
            qc.h(0)
        qc.measure(0, 0)

        circuitos.append(qc)

    qc_transpiled = transpile(circuitos, backend)
    job = backend.run(qc_transpiled, shots=1, memory=True)
    resultados = job.result()

    #Recopilacion y cribado
    clave_alice = []
    clave_bob = []

    for i in range(longitud_clave):
        resultado_bob = int(resultados.get_memory(i)[0])


        if resultado_bob == 1:
            # Deducción del bit según la base de Bob
            if bob_bases[i] == 0:      # Midió en Z, obtuvo |1> → solo posible si Alice envió |+>
                bit_deducido = 1
            else:                      # Midió en X, obtuvo |-> → solo posible si Alice envió |0>
                bit_deducido = 0

            clave_alice.append(alice_bits[i])
            clave_bob.append(bit_deducido)

    # Conversión a arrays para cálculos
    clave_alice = np.array(clave_alice)
    clave_bob = np.array(clave_bob)

    errores = np.sum(clave_alice != clave_bob)
    total_bits_utiles = len(clave_alice)
    qber = (errores / total_bits_utiles) * 100 if total_bits_utiles > 0 else 0.0

    # Impresión de resultados (similar a BB84 y Six-State)
    print("="*60)
    print(f" SIMULACIÓN B92 | Presencia de Eve: {con_eve}")
    print("="*60)
    print(f"Longitud de transmisión inicial: {longitud_clave} fotones")
    print(f"Longitud de clave cribada: {total_bits_utiles} bits (~25% esperado)")
    print(f"Discrepancias detectadas: {errores} bits")
    print(f"Tasa de Error Cuántico (QBER): {qber:.2f}%")

    return qber

# =============== Main ===================
print("Protocolo B92...")
b92_limpio = b92(longitud_clave=1000, con_eve=False)
print("\n")
b92_interceptado = b92(longitud_clave=1000, con_eve=True)

Protocolo B92...
 SIMULACIÓN B92 | Presencia de Eve: False
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 234 bits (~25% esperado)
Discrepancias detectadas: 0 bits
Tasa de Error Cuántico (QBER): 0.00%


 SIMULACIÓN B92 | Presencia de Eve: True
Longitud de transmisión inicial: 1000 fotones
Longitud de clave cribada: 377 bits (~25% esperado)
Discrepancias detectadas: 123 bits
Tasa de Error Cuántico (QBER): 32.63%
